> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t02_bronze_to_silver_initial` · notebook `02_bronze_to_silver` · started 2026-10-11 12:59 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `mode=incremental`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 02 · Bronze → Silver
Casts epoch integers to timestamps, decodes titles, removes PII (drops names/avatars, hashes user ids with a
salted SHA-256), derives body features, applies data-quality rules and upserts every Silver table with
**`MERGE INTO`**. Re-running the same batches changes nothing (idempotent).

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: Bronze batches not yet in Silver | `backfill`: re-process the selected batches |
| `batch_ids` | empty | comma-separated batch ids |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `pii_salt` | empty (read from secret `so_lakehouse/pii_salt`) | same |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("pii_salt", "")

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.silver import run_bronze_to_silver

cfg = build_config(need_salt=True)
audit = run_bronze_to_silver(
    spark, cfg,
    mode=widget("mode"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    fail_on_error=True,
)

Bronze-to-Silver: processing 5 batch(es): full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z
[SUCCESS] Bronze-to-Silver | questions                          | read=7677 ins=7618 upd=0 del=0 quar=0 | 25.5s | 7618 unique questions after de-duplication
[SUCCESS] Bronze-to-Silver | question_tags                      | read=24177 ins=24177 upd=0 del=0 quar=0 | 13.4s
[SUCCESS] Bronze-to-Silver | answers                            | read=9701 ins=9611 upd=0 del=0 quar=0 | 8.4s | 9611 unique answers after de-duplication
[SUCCESS] Bronze-to-Silver | users                              | read=17280 ins=8332 upd=0 del=0 quar=0 | 7.8s
[SUCCESS] Bronze-to-Silver | question_snapshots                 | read=7385 ins=7385 upd=0 del=0 quar=0 | 4.8s
[SUCCESS] Bronze-to-Silver | question_metrics (from snapshots)  | read=7285 ins=0 upd=7282 del=0 quar=0 | 9.1s
[SUCCESS] Bronze-to-Silver | deleted_questions -> questions     | read=1 in

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
2e5bde31-3cea-4e75-9c95-313453d6db0f,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,questions,,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.questions,2026-10-11T12:59:35.140Z,2026-10-11T13:00:00.598Z,25.458039,SUCCESS,7677,7618,0,0,0,7618 unique questions after de-duplication,2026-10-11T13:00:00.598Z
3520af65-31d8-4fd3-84cf-65fd4479ba5e,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,question_tags,,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.question_tags,2026-10-11T13:00:04.066Z,2026-10-11T13:00:17.482Z,13.415917,SUCCESS,24177,24177,0,0,0,,2026-10-11T13:00:17.482Z
239d9e68-736c-4704-a3e9-29938ff0ed41,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,answers,,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.answers,2026-10-11T13:00:19.073Z,2026-10-11T13:00:27.444Z,8.371009,SUCCESS,9701,9611,0,0,0,9611 unique answers after de-duplication,2026-10-11T13:00:27.444Z
3fa9c0a8-79c5-434c-bf77-2a96a86d3487,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,users,,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.users,2026-10-11T13:00:29.361Z,2026-10-11T13:00:37.116Z,7.755158,SUCCESS,17280,8332,0,0,0,,2026-10-11T13:00:37.116Z
e29702ff-f790-433a-b9c1-f18b3e2f07d3,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,question_snapshots,,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.question_snapshots,2026-10-11T13:00:38.503Z,2026-10-11T13:00:43.310Z,4.80712,SUCCESS,7385,7385,0,0,0,,2026-10-11T13:00:43.310Z
ba4ae95a-11a5-44c1-9cc9-e6e75311fbc8,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,question_metrics (from snapshots),,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.questions,2026-10-11T13:00:44.653Z,2026-10-11T13:00:53.776Z,9.122493,SUCCESS,7285,0,7282,0,0,,2026-10-11T13:00:53.776Z
eb5b04de-644c-475d-a57f-e7ffeff86e8e,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,deleted_questions -> questions,,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.questions,2026-10-11T13:00:55.194Z,2026-10-11T13:01:05.165Z,9.971817,SUCCESS,1,0,1,0,0,soft delete (is_deleted = true),2026-10-11T13:01:05.166Z
fcf440e1-b4bd-45bd-a882-0380aae8e61a,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Silver,incremental,mixed,deleted_questions -> answers,,"unprocessed batches -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z]",workspace.so_silver.answers,2026-10-11T13:01:06.781Z,2026-10-11T13:01:14.885Z,8.103901,SUCCESS,1,0,2,0,0,soft delete (is_deleted = true),2026-10-11T13:01:14.885Z
f561a15c-6ce4-4d68-bd04-1c965e01c310,bronze_to_silver-20261011T125933Z-d54ec7,bronze_to_silver,Bronze-to-Si